# YOLO : You Only Look Once


Object detection asks two questions:
1. What objects are present ?
2. What exactly are they ?

so the models needs to output:

```
Dog → [x,y,width,height]
cat → [x,y,width,height]
car → [x,y,width,height]
```

The rectangle around an object is called a bounding box.

Object detection is "tell me WHAT is there and WHERE it is"

Classification= WHAT
<br>
Detection = WHAT + WHERE

Thats the fundamental problem YOLO solves.


Before YOLO, Detection pipeline was in multiple steps:

```
Example:
Image
 ↓
Find possible regions
 ↓
Run CNN on each region
 ↓
Classify regions
 ↓
Improve boxes
 ↓
Remove duplicate boxes
 ↓
Final detections
```


YOLO`s ideas is radically simple:
```
Image
 ↓
ONE neural network
 ↓
boxes + classes

Hence, YOLO = You Only Look Once
```

The entire image is processed once, and the network predicts object locations and classes simultaneously.

<br>

### Before YOLO:

**R-CNN**: First find interesting regions, then classify each region.

pipeline:
```
Image
 ↓
Selective Search
 ↓
Many candidate regions
 ↓
CNN
 ↓
Classification

```
RNN is essentially:  Region proposal → CNN classification

It is slow because we are repeatedly processing many regions.


**Faster R-CNN**: It improved R-CNN and important improvements were:

  - uses ROI pooling to extract features for regions of interest
  - Uses multi-task loss to train classifications and localization together
  - Avoids storing CNN features on disk
  - Moves toward end to end training.

  > Instead of running the CNN seperately for every region, extract image feature once and reuse them. Must faster but it still depends on regions proposals.


  Faster R-CNN asks "why do we need selective search at all ?" -> we dont. Lets nmake neural network generate region proposal itself.


It introduces: `RPN - Region Proposal Network` : These areas of the image probably contain objects.


```
Image
 ↓
CNN features
 ↓
RPN
 ↓
Candidate regions
 ↓
Fast R-CNN
 ↓
Classes + refined boxes

```

So Faster R-CNN removes the external Selective Search algorithm.



so the evolution is

```

R-CNN
Region proposals + CNN

        ↓

Fast R-CNN
Better feature sharing + ROI pooling

        ↓

Faster R-CNN
Neural network generates region proposals


```

But there is still a proposal stage, YOLO goes one step further.

### **Feature Pyramid Network - FPN**

A single feature scale may not be good at detecting both. FNP creates at multiple scales.

```
Image
 ↓
Downsample
 ↓
small/high-level feature maps
 ↓
Upsample
 ↓
combine with earlier features
 ↓
predictions at multiple scales


```

> use information from multiple resolutions to detect objects of different sizes. The skip connections helps preserve useful fine grained information.

### YOLO
> " forget the complicated proposal pipeline. Give the whole image,  and i will predict everything in one pass"

This is called single stage detector.

idea:

```
IMAGE
  ↓
CNN
  ↓
S × S grid
  ↓
Bounding boxes + confidence + classes
```
This makes YOLO extremely fast.

YOLOv1 achieved roughly 45 FPS, which was a major breakthrough for real-time object detection.


YOLO idea: turn detection into regression

Normally, object detects sounds like complicated problems
> Find objects → classify them → locate them → remove duplicates

YOLO reframes it as:
Predict numbers


Thats a regression problem. The netework receives an image and directly predicts:

```
x
y
w
h
confidence
class probabilities

```

so:
> Image -> numbers describing objects


### The S * S grid

Yolo divides the image into grid.

For YOLOv1 :  7 × 7 grid,so there are 49 cells. The  grid cell containing the center of an object is responsible for detecting that object.

let
```
+---+---+---+
|   |   |   |
+---+---+---+
|   | 🐶|   |
+---+---+---+
|   |   |   |
+---+---+---+
```
The dog is inside the middle cell, therefore middle cell is responsible for predicting the dog.  This rule is fundamental to YOLOv1.

Each cell predicts : Bounding boxes:
- each box contains $[x,y,w,h, confidence]$, 5 numbers .
  * If there are B boxes then $B*5$

  * class probabilities: There are C classes:
  ```
  P(class₁ | object)
  P(class₂ | object)
  ...
  P(classC | object)
  ```
  so output per cell is:
  B × 5 + C

  and the entire image produces: S × S × (B × 5 + C)

> YOLO output= grid * (box information + class information)

- Bounding box coordinates: Each bounding box has (x,y,w,h)
  * where : x = center x-coordinate
  * y = center y-coordinate
  * w= width
  * h =height

  The x,y coordinates are relative to the grid cell. The width and height are normalized relative to image.

  > The objects center is here,  and its rectangle is this wide and this tall.

- **Confidence** : Important equation
  * The confidence is : $confidence= P(object)*IoU$
  where,
    * P(object) = Probability that object exists
    * I0U (intersection over union)  = how much predicted and true boxes overlaps
    $iou=Area of overlap/ Area of union$
    
    when IoU ≈ 1, they overlap almost perfectly. If they barely overlap: IoU ≈ 0

Confidence asks : "is there an object here, and did i draw its box correctly?"

- **class probabilities** : Each cell also predicts $P(classs^i | object)$
> "assuming there is an object here, whats the probability that its class i ?"

- Class Specific Confidence
  At inference time, YOLO combines:
   * class probability × box confidence
    
    so:
   * P(classi|object)*IoU which becomes $P(class^i)*IoU$

   "How likely is this specific class, multiplied by how good the box is ?"

   








## YOLOv1 output

Formula : $S*S*(B*5+C)$

where:
  * S=grid size
  * B=boxes per cell
  * 5= x,y,w,h, confidence
  * C=number of classes

 Yolo essentially learns three things:
 ```
              YOLO LEARNS
                  │
       ┌──────────┼──────────┐
       ↓          ↓          ↓
   LOCATION    OBJECT?     CLASS?
   x,y,w,h     confidence   dog/cat/...

   ```

   Therefore:
   $Loss = localization + Confidence + Classification$

   Localization:
   > Your box is in the wrong place.

   Confidence:
   > " You said there`s an object here, but there isnt"

   Classification:
   > You found an object, but called the dog a cat


### Yolov1 uses  λcoord=5:
Localization is important.  A badly positioned box is serious error. so YOLOv1 says:

Λcoord=5

meaning:
> Pay extra attention to getting the box coordinate right.



Why λnoobj = 0.5

Most grid cells contain nothing.

For example:
```
49 cells

maybe only 3 contain objects

46 = background

```
if every background error had equal importance, background would dominate training. Therefore, λnoobj​=0.5

> Dont let all the empty cells overwhelm the actual projects.


## why $\sqrt{width}$  and $\sqrt{height}$

let:

```
True width = 100
Prediction = 110

error = 10

versus:

True width = 10
Prediction = 20

error = 10

```
Same absolute error. But the second mistake is more significant relative to the object. YOLOV1 therefore uses: $\sqrt{w} , \sqrt{h}$
in localization loss

> Be more sensitive to size errors on small objects


### which box gets responsibility ?

Suppose two boxes the predicts same object:
```
Box A → IoU = 0.72
Box B → IoU = 0.31
```

Yolo chooses:

```
Box A → responsible
Box B → not responsible
```
The highest-IoU predictor gets responsibility.
> The box that already fits best learn  to represent the object.


### Inference

After training:

```
IMAGE
  ↓
YOLO
  ↓
many predictions
  ↓
remove weak predictions
  ↓
NMS
  ↓
FINAL BOXES
```


### NMS
Yolo can predict
```

Dog box A → 0.95
Dog box B → 0.90
Dog box C → 0.70

```
but they are same dog.

NMS says:
> keep the strongest box and remove highly overlapping duplicates.

```
0.95 ← KEEP
0.90 ← REMOVE
0.70 ← REMOVE
```

so:
> NMS = duplicate cleanup


### FPN
objects have different sizes:

```
🐘 large
🐕 medium
🐜 tiny
```

one resolution isnt ideal for everything. FPN gives the detector information at multiple scales.

> Coarse features help the big objects; fine features help with small objects.



<br>

----


### YOLO evolution
| Version | Mental hook                          |
| ------- | ------------------------------------ |
| **v1**  | One network predicts everything      |
| **v2**  | Anchors + better localization        |
| **v3**  | Multi-scale detection                |
| **v4**  | Speed + accuracy + training tricks   |
| **v5**  | PyTorch + auto-anchors + model sizes |
| **v6**  | Industrial deployment + efficiency   |
| **v7**  | Better training + scaling            |
| **v8**  | Anchor-free + multiple vision tasks  |
| **v9**  | PGI + GELAN / information flow       |


----
> "YOLO receives the whole image and processes it with one neural network. In YOLOv1, the image is divided into a grid. The cell containing the dog's center becomes responsible for detecting it. The network predicts the dog's box — x, y, width, height — its confidence that an object exists there, and class probabilities. During training, the model is penalized for incorrect box coordinates, object confidence, and class predictions. During inference, weak predictions are removed and NMS removes duplicate overlapping boxes. The result is the dog's location and class."


```
                    OBJECT DETECTION
                           │
                  ┌────────┴────────┐
                  ↓                 ↓
                WHAT?             WHERE?
             classification    localization
                  │                 │
                  └────────┬────────┘
                           ↓
                          YOLO
                           │
                     ONE CNN PASS
                           │
                       S × S GRID
                           │
            ┌──────────────┼──────────────┐
            ↓              ↓              ↓
           x,y            w,h        confidence
                                          │
                                   P(object) × IoU
                           │
                    class probabilities
                           ↓
                  confidence filtering
                           ↓
                          NMS
                           ↓
                  FINAL DETECTIONS


```

